In [13]:
import pandas as pd
import numpy as np
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# 1. DATA FILTERING (FIRST ROUND ONLY)
df = pd.read_csv('men_2026_matchups_training.csv')
df_first_round = df[df['round'] == 'First Round'].copy()

# Features Pool (49 variables)
features_49 = [
    '3man_bpm', '5man_bpm', 'torvik_rtg', 'wab', '5man_dbpm', '3man_dbpm',
    'experience_weighted_production', '5man_dprpg', '3man_prpg', '3man_dprpg',
    '5man_prpg', 'def_lineup_depth_quality', 'lineup_depth_quality',
    'def_experience_impact', 'off_close2_fg_pct', 'bench_scoring_ratio',
    'rotation_balance', 'def_3pt_fg_pct', 'tempo_advantage', 'off_far2_share',
    'four_factors_composite', 'effective_possession_rate', 'def_four_factors_composite',
    'def_effective_possession_rate', 'def_close2_fg_pct', 'size', 'def_far2_share',
    'kenpom_off', 'def_assist_suppression', 'assist_to_usage_ratio', 'def_dunk_share',
    'off_3pt_fg_pct', 'ft_pct', 'perimeter_efficiency', 'def_rim_efficiency',
    'efgd_pct', 'def_close2_share', 'def_rim_to_three_ratio', 'def_paint_touch_rate',
    'rim_to_three_ratio', 'off_dunk_share', 'def_perimeter_efficiency',
    'def_size_speed_index', 'size_speed_index', 'block_efficiency',
    'def_block_efficiency', 'orb_pct', 'ftr', 'drb_pct'
]

X = df_first_round[features_49].fillna(df_first_round[features_49].median())
y = df_first_round['win']

# 2. SPLITS & SCALING
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 3. FEATURE SELECTION (TOP 25 FROM 49)
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    max_features=25,
    threshold=-np.inf
)
X_train_sel = selector.fit_transform(X_train_scaled, y_train)
X_val_sel = selector.transform(X_val_scaled)
selected_names = np.array(features_49)[selector.get_support()]

# 4. OPTIMIZATION STUDY
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'max_depth': trial.suggest_int('max_depth', 5, 18),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 6),
        'ccp_alpha': trial.suggest_float('ccp_alpha', 1e-6, 5e-4, log=True),
        'max_features': 'sqrt',
        'random_state': 42,
        'n_jobs': -1
    }
    rf = RandomForestClassifier(**params)
    return cross_val_score(rf, X_train_sel, y_train, cv=5, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

# 5. OUTPUT BEST HYPERPARAMETERS
print("\n" + "="*30)
print("BEST HYPERPARAMETERS FOR FIRST ROUND MODEL")
print("="*30)
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print("="*30)
print(f"Best CV Mean: {study.best_value:.4f}")

# Final Validation Check
best_rf = RandomForestClassifier(**study.best_params, random_state=42)
best_rf.fit(X_train_sel, y_train)
val_acc = accuracy_score(y_val, best_rf.predict(X_val_sel))
print(f"Validation Accuracy: {val_acc:.4f}")

[I 2026-03-19 01:20:46,773] A new study created in memory with name: no-name-1a3d2af7-a8ab-4627-a780-c9f62a07bd3b
[I 2026-03-19 01:21:06,722] Trial 0 finished with value: 0.7569633787757284 and parameters: {'n_estimators': 650, 'max_depth': 12, 'min_samples_split': 9, 'min_samples_leaf': 4, 'ccp_alpha': 4.129070558140541e-05}. Best is trial 0 with value: 0.7569633787757284.
[I 2026-03-19 01:21:17,014] Trial 1 finished with value: 0.7524191392675755 and parameters: {'n_estimators': 457, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 3, 'ccp_alpha': 2.0760525305435515e-06}. Best is trial 0 with value: 0.7569633787757284.
[I 2026-03-19 01:21:22,492] Trial 2 finished with value: 0.7430366212242715 and parameters: {'n_estimators': 449, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 5, 'ccp_alpha': 5.684828014110924e-05}. Best is trial 0 with value: 0.7569633787757284.
[I 2026-03-19 01:21:31,248] Trial 3 finished with value: 0.743090082865544 and parameters: {'n_es


BEST HYPERPARAMETERS FOR FIRST ROUND MODEL
n_estimators: 385
max_depth: 15
min_samples_split: 2
min_samples_leaf: 5
ccp_alpha: 1.734255480280635e-06
Best CV Mean: 0.7570
Validation Accuracy: 0.7053


In [14]:
# Print the 25 variables that survived the selection
selected_names = np.array(features_49)[selector.get_support()]
print("Variables used in the final model:")
print(selected_names)

Variables used in the final model:
['3man_bpm' '5man_bpm' 'torvik_rtg' 'wab' '5man_dbpm' '3man_dbpm'
 'experience_weighted_production' '5man_dprpg' '3man_prpg' '3man_dprpg'
 '5man_prpg' 'def_lineup_depth_quality' 'lineup_depth_quality'
 'def_experience_impact' 'rotation_balance' 'def_3pt_fg_pct'
 'tempo_advantage' 'four_factors_composite' 'def_close2_fg_pct' 'size'
 'def_far2_share' 'kenpom_off' 'off_3pt_fg_pct' 'def_size_speed_index'
 'size_speed_index']
